# 06 — Hybrid search

**Scenario.** Someone asks about `WB-8841`. That is an ID, not a paraphrase. Dense search (vectors) is good at meaning. **BM25** is good at exact tokens. **RRF** merges the two ranked lists.

1. Rebuild the index.
2. Compare dense vs BM25 vs fused ranks for an ID query.
3. Inspect the reranking features that produce the final order.

Definitions:

- Dense retrieval ranks by vector similarity.
- BM25 ranks by token overlap and rarity.
- RRF combines ranked lists without trusting raw score scales.

Edge cases:

- Dense search may find related watch-book rows but miss the exact entry.
- BM25 may over-rank prompt-injection text because it repeats the user's words.
- Fusion improves recall, but security filters must still run after retrieval.


Definitions and context:

| Concept | Meaning |
| --- | --- |
| Sparse retrieval | Search using token statistics such as BM25 |
| Recall | Whether the right evidence appears somewhere in the candidates |
| Precision | Whether the top results are mostly useful |
| RRF | Reciprocal Rank Fusion, a rank-based merge method |
| Reranking | Reordering candidates with extra signals |

Hybrid retrieval is common because dense and sparse search fail differently.
Dense search handles paraphrases; BM25 handles exact strings. RRF is simple and
stable because it uses ranks instead of mixing incompatible score scales.

Similarity search techniques:

| Technique | Good for | Weakness |
| --- | --- | --- |
| Cosine over embeddings | Meaning and paraphrases | Weak for rare exact IDs |
| Dot product | Fast when vectors are normalized consistently | Scale-sensitive if vectors are not normalized |
| BM25 | Exact terms, codes, policy IDs | Weak for synonyms |
| Metadata filter | Classification, date, status, source type | Does not rank relevance by itself |
| Cross-encoder rerank | Strong final relevance scoring | Slower and usually another model call |
| Heuristic rerank | Recency, status, term overlap, source trust | Needs careful tuning |


In [ ]:
import _course
from lib.pipeline import build_index, cosine, embed, rerank, rrf, tokenize

index = build_index()
q = "What is Watch Book entry WB-8841?"
vec = embed([q])[0]
dense = index.dense(vec, 6)
sparse = index.sparse(q, 6)

print("DENSE")
for h in dense[:5]:
    print(f"  {h.dense_score:.3f}  {h.chunk.doc_id:22} {h.chunk.section}")
print("BM25")
for h in sparse[:5]:
    print(f"  {h.sparse_score:.3f}  {h.chunk.doc_id:22} {h.chunk.section}")
print("RRF + recency")
for h in rerank(q, rrf([dense, sparse]), keep=5):
    print(f"  {h.rerank_score:.3f}  {h.chunk.status:11} {h.chunk.doc_id} / {h.chunk.section}")


In [ ]:
# Manual dense similarity search: embed every chunk and compute cosine.
chunks = list(index.chunks.values())
chunk_vectors = embed([c.embedded_text for c in chunks])
manual_dense = sorted(
    ((cosine(vec, ch_vec), chunk) for ch_vec, chunk in zip(chunk_vectors, chunks)),
    reverse=True,
    key=lambda item: item[0],
)[:5]

print("MANUAL COSINE")
for score, chunk in manual_dense:
    print(f"  {score:.3f}  {chunk.doc_id:22} {chunk.section}")


In [ ]:
# Manual BM25 intuition: tokenize the question and inspect exact terms.
print("query tokens:", tokenize(q))
print()
print("BM25 is strong here because WB-8841 is an exact token in the corpus.")
for h in sparse[:5]:
    snippet = " ".join(h.chunk.text.split())[:120]
    print(f"  {h.sparse_score:.3f}  {h.chunk.doc_id:22} {snippet}")


In [ ]:
# RRF gives each result credit based on rank position:
# contribution = 1 / (k + rank). k=60 is common.
fused = rrf([dense, sparse], k=60)
print("RRF candidates")
for h in fused[:8]:
    print(
        f"  fused={h.fused_score:.4f} dense={h.dense_score:.3f} "
        f"bm25={h.sparse_score:.3f} {h.chunk.doc_id} / {h.chunk.section}"
    )


In [ ]:
# Final rerank adds status, recency, lexical overlap, and score signals.
ranked = rerank(q, fused, keep=6)
print("FINAL RERANK")
for h in ranked:
    print(
        f"  final={h.rerank_score:.3f} status={h.chunk.status:11} "
        f"class={h.chunk.classification:10} {h.chunk.doc_id} / {h.chunk.section}"
    )


This list is not yet safe: an advisor could still see MNPI if it ranked. Next: `07-stale-documents.ipynb`.
